In [0]:
%run ./00_config

### Create the folder structure on ADLS using dbutils

In [0]:
# Create project folder structure in ADLS

folders = [
    "landing/nyc_taxi/yellow/",
    "landing/nyc_taxi/zones/",
    "landing/weather/",
    "landing/events/",
    "landing/synthetic/live_ride_requests/",

    "checkpoints/bronze/",
    "checkpoints/silver/",
    "checkpoints/gold/",
    "checkpoints/streaming/",

    "quarantine/taxi_trips/",

    "curated/bronze/yellow_taxi_trips/",
    "curated/bronze/taxi_zones/",
    "curated/bronze/weather/",
    "curated/bronze/events/",
    "curated/bronze/live_ride_requests/",

    "curated/silver/trips_clean/",
    "curated/silver/taxi_zones/",
    "curated/silver/weather_clean/",
    "curated/silver/events_clean/",
    "curated/silver/live_ride_requests_clean/",

    "curated/gold/zone_hour_demand_baseline/",
    "curated/gold/weather_demand_impact/",
    "curated/gold/event_demand_impact/",
    "curated/gold/live_demand_by_zone/",
    "curated/gold/live_demand_anomaly/",
]

for folder in folders:
    path = f"{BASE_PATH}/{folder}"
    dbutils.fs.mkdirs(path)
    print(f"Created or exists: {path}")

   
### Download Trips data from TLC to ADLS landing (Jan 2020 – present)

In [0]:
# Downloads Yellow Taxi trip Parquet files from the NYC TLC public dataset.
#
# Dynamic date range: 2020-01 through the current month.
# TLC publishes with ~2 month lag, so recent months may 404 (handled gracefully).
#
# Idempotent: files already present in the landing zone are skipped.
# Safe to re-run monthly on a schedule — only new months are downloaded.

import os
import requests
from datetime import datetime

volume_landing_root = "/Volumes/nyc_taxi_company/bronze/landing_files"
taxi_volume_landing_path = f"{volume_landing_root}/nyc_taxi/yellow"

# Browser-like User-Agent to avoid CDN 403 blocks on python-requests default UA
HTTP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

# --- Dynamic date range: 2020-01 through current month ---
now = datetime.now()
START_YEAR = 2020

months = []
for year in range(START_YEAR, now.year + 1):
    end_month = now.month if year == now.year else 12
    for month in range(1, end_month + 1):
        months.append(f"{year}-{month:02d}")

print(f"Months to process: {len(months)} ({months[0]} → {months[-1]})")

source_base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data"

downloaded = 0
skipped = 0
failed = []

for i, month in enumerate(months, 1):
    year_str = month[:4]
    month_str = month[5:7]
    filename = f"yellow_tripdata_{month}.parquet"
    url = f"{source_base_url}/{filename}"

    target_dir = f"{taxi_volume_landing_path}/year={year_str}/month={month_str}"
    target_file = f"{target_dir}/{filename}"

    # Idempotent: skip files already present
    if os.path.exists(target_file) and os.path.getsize(target_file) > 0:
        skipped += 1
        continue

    os.makedirs(target_dir, exist_ok=True)
    print(f"[{i}/{len(months)}] Downloading: {filename}")

    try:
        with requests.get(url, headers=HTTP_HEADERS, stream=True, timeout=300) as response:
            response.raise_for_status()
            with open(target_file, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        size_mb = os.path.getsize(target_file) / (1024 * 1024)
        print(f"  Done: {size_mb:.1f} MB")
        downloaded += 1
    except requests.exceptions.HTTPError:
        # 403/404 = not published yet (TLC CDN returns either depending on age)
        if response.status_code in (403, 404):
            print(f"  Not available yet ({response.status_code}) - expected for recent months")
            failed.append((month, str(response.status_code)))
        else:
            print(f"  HTTP error {response.status_code}")
            failed.append((month, str(response.status_code)))
    except Exception as e:
        print(f"  Error: {e}")
        failed.append((month, str(e)))

print(f"\n{'='*60}")
print(f"Downloaded: {downloaded} | Skipped (exist): {skipped} | Failed: {len(failed)}")
if failed:
    print(f"Failed months: {[m for m, _ in failed]}")

In [0]:
# Validate expanded taxi data in landing zone
import os

taxi_volume_landing_path = "/Volumes/nyc_taxi_company/bronze/landing_files/nyc_taxi/yellow"

years_summary = {}
for year_dir in sorted(os.listdir(taxi_volume_landing_path)):
    if not year_dir.startswith("year="):
        continue
    year = year_dir.split("=")[1]
    year_path = os.path.join(taxi_volume_landing_path, year_dir)
    month_count = 0
    total_size_mb = 0
    for month_dir in sorted(os.listdir(year_path)):
        if not month_dir.startswith("month="):
            continue
        month_path = os.path.join(year_path, month_dir)
        for f in os.listdir(month_path):
            if f.endswith(".parquet"):
                total_size_mb += os.path.getsize(os.path.join(month_path, f)) / (1024 * 1024)
                month_count += 1
    years_summary[year] = {"months": month_count, "size_mb": round(total_size_mb, 1)}

for year, info in sorted(years_summary.items()):
    print(f"Year {year}: {info['months']} months, {info['size_mb']:.1f} MB")

total_files = sum(v['months'] for v in years_summary.values())
total_size = sum(v['size_mb'] for v in years_summary.values())
print(f"\nTotal: {total_files} monthly files, {total_size:.1f} MB ({total_size/1024:.1f} GB)")

### Download Taxi lookup zones data from TLC into ADLS

In [0]:
# Downloads TLC taxi zone lookup CSV (one-time, static reference data).
# Physical ADLS target:
#   abfss://taxicompany@azinfrasummit.dfs.core.windows.net/landing/nyc_taxi/zones/taxi_zone_lookup.csv

import os
import requests

volume_landing_root = "/Volumes/nyc_taxi_company/bronze/landing_files"

# Browser-like User-Agent to avoid CDN 403 blocks
HTTP_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_target_dir = f"{volume_landing_root}/nyc_taxi/zones"
zone_target_file = f"{zone_target_dir}/taxi_zone_lookup.csv"

os.makedirs(zone_target_dir, exist_ok=True)

# Idempotent: skip if already downloaded
if os.path.exists(zone_target_file) and os.path.getsize(zone_target_file) > 0:
    print(f"SKIP (exists): {zone_target_file}")
else:
    print(f"Downloading {zone_lookup_url}")
    with requests.get(zone_lookup_url, headers=HTTP_HEADERS, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(zone_target_file, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print(f"Done: {zone_target_file}")

In [0]:
#validate

display(dbutils.fs.ls(f"{LANDING_PATH}/nyc_taxi/zones/"))

   
### Download NOAA daily weather data for NYC Central Park (2020 – present)

In [0]:
# Downloads NOAA GHCND Daily Summaries for NYC Central Park station.
# One JSON file per year: noaa_daily_central_park_{year}.json
#
# Dynamic date range: 2020 through current year.
#
# Idempotent behavior:
#   - Completed past years: skipped (immutable historical data)
#   - Current year: ALWAYS re-fetched (incomplete, grows daily)
#
# NOAA CDO API constraints:
#   - Max 1 year per request (startdate to enddate)
#   - Max 1000 records per response (paginate with offset)
#   - Rate limit: 5 requests/second
#   - Response shape: {"metadata": {"resultset": {"count": N}}, "results": [...]}
#
# Station: GHCND:USW00094728 = NY CITY CENTRAL PARK, NY US
# Data types: PRCP (precipitation mm), SNOW (mm), TMAX (°C), TMIN (°C)
# Expected: ~1,460 records/year (365 days × 4 data types)

import os
import json
import requests
import time
from datetime import datetime

NOAA_TOKEN = dbutils.secrets.get(scope="weather-data", key="NOAA-CDO")

volume_landing_root = "/Volumes/nyc_taxi_company/bronze/landing_files"
weather_target_dir = f"{volume_landing_root}/weather"
os.makedirs(weather_target_dir, exist_ok=True)

base_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
headers = {"token": NOAA_TOKEN}

# --- Dynamic date range: 2020 through current year ---
now = datetime.now()
START_YEAR = 2020

year_ranges = []
for year in range(START_YEAR, now.year + 1):
    start = f"{year}-01-01"
    # Current year: only up to today (future dates have no data)
    end = now.strftime("%Y-%m-%d") if year == now.year else f"{year}-12-31"
    year_ranges.append((start, end))

# Current year is always incomplete — delete and re-fetch to capture recent days.
# Past years are immutable and skipped via the exists check below.
current_year_file = f"{weather_target_dir}/noaa_daily_central_park_{now.year}.json"
if os.path.exists(current_year_file):
    os.remove(current_year_file)
    print(f"Refreshing current year: deleted noaa_daily_central_park_{now.year}.json\n")

# Set REFETCH_ALL = True to re-download all years (e.g. after fixing a bug).
# After a clean run, set back to False so future runs skip completed years.
REFETCH_ALL = False

if REFETCH_ALL:
    for year in range(START_YEAR, now.year):
        f = f"{weather_target_dir}/noaa_daily_central_park_{year}.json"
        if os.path.exists(f):
            os.remove(f)
    print("REFETCH_ALL=True: deleted all year files for re-download\n")

total_records = 0

for start_date, end_date in year_ranges:
    year = start_date[:4]
    target_file = f"{weather_target_dir}/noaa_daily_central_park_{year}.json"

    # Past years: skip if already downloaded
    if os.path.exists(target_file) and os.path.getsize(target_file) > 0:
        print(f"SKIP (exists): noaa_daily_central_park_{year}.json")
        continue

    all_results = []
    offset = 1

    while True:
        params = {
            "datasetid": "GHCND",
            "stationid": "GHCND:USW00094728",
            "startdate": start_date,
            "enddate": end_date,
            "datatypeid": ["PRCP", "SNOW", "TMAX", "TMIN"],
            "units": "metric",
            "limit": 1000,
            "offset": offset,
        }

        response = requests.get(base_url, params=params, headers=headers, timeout=120)
        response.raise_for_status()

        payload = response.json()
        results = payload.get("results", [])
        all_results.extend(results)

        # NOAA nests pagination info under metadata.resultset (NOT top-level)
        # {"metadata": {"resultset": {"offset": 1, "count": 1460, "limit": 1000}}}
        resultset = payload.get("metadata", {}).get("resultset", {})
        total_count = resultset.get("count", 0)

        if offset + 999 >= total_count or len(results) == 0:
            break

        offset += 1000
        time.sleep(0.3)  # Rate limit courtesy

    print(f"  {year}: {len(all_results)} records ({start_date} -> {end_date})")
    total_records += len(all_results)

    with open(target_file, "w") as f:
        for row in all_results:
            f.write(json.dumps(row) + "\n")

    time.sleep(0.3)

# One-time cleanup: remove legacy Q1-only file from initial bootstrap
old_q1_file = f"{weather_target_dir}/noaa_daily_central_park_2024_q1.json"
if os.path.exists(old_q1_file):
    os.remove(old_q1_file)
    print(f"\nRemoved legacy file: noaa_daily_central_park_2024_q1.json")

print(f"\nWeather data complete: {total_records} new records across {len(year_ranges)} year chunks")

In [0]:
# Validate expanded weather data in landing zone

display(dbutils.fs.ls(f"{LANDING_PATH}/weather/"))

# Quick row count per file
import os
weather_dir = "/Volumes/nyc_taxi_company/bronze/landing_files/weather"
for f in sorted(os.listdir(weather_dir)):
    if f.endswith(".json"):
        filepath = os.path.join(weather_dir, f)
        with open(filepath) as fh:
            lines = sum(1 for _ in fh)
        print(f"{f}: {lines} records")

   
### Download/curate NYC sports events (2020 – present) into ADLS landing

In [0]:
# Curates NYC sports event schedules from ESPN and MLB Stats APIs.
#
# Dynamic date range: 2020 through end of current year.
# Always regenerates the FULL file — schedules change due to postponements,
# cancellations, and newly announced games. Atomic replace ensures consistency.
#
# Output: landing/events/sports_events.json  (stable name, overwritten each run)
#
# Leagues:
#   NBA  – Knicks (MSG), Nets (Barclays Center)
#   NHL  – Rangers (MSG)
#   MLB  – Yankees (Yankee Stadium), Mets (Citi Field)
#
# NOTE: After running, trigger a FULL REFRESH of the SDP pipeline (not a normal
# trigger). Auto Loader won't reprocess a file at the same path by default.
# Alternatively, set cloudFiles.allowOverwrites=true on the events streaming table.

import os
import json
import time
import requests
import hashlib
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

volume_landing_root = "/Volumes/nyc_taxi_company/bronze/landing_files"

# --- Stable file paths (no date in name — always overwritten) ---
sports_target_dir = f"{volume_landing_root}/events"
sports_tmp_dir = f"{sports_target_dir}/_tmp"

sports_tmp_file = f"{sports_tmp_dir}/sports_events.json"
sports_final_file = f"{sports_target_dir}/sports_events.json"

sports_adls_tmp_file = f"{LANDING_PATH}/events/_tmp/sports_events.json"
sports_adls_final_file = f"{LANDING_PATH}/events/sports_events.json"

os.makedirs(sports_tmp_dir, exist_ok=True)

# Clean up any previous temp file
try:
    dbutils.fs.rm(sports_adls_tmp_file)
except Exception:
    pass

NY_TZ = ZoneInfo("America/New_York")

# --- Dynamic date range: 2020-01-01 through end of current year ---
now = datetime.now()
START_YEAR = 2020
DATE_START_LOCAL = datetime(START_YEAR, 1, 1, 0, 0, 0, tzinfo=NY_TZ)
DATE_END_LOCAL = datetime(now.year + 1, 1, 1, 0, 0, 0, tzinfo=NY_TZ)

# ESPN season param = end year (e.g. 2026 = 2025-26 season)
ESPN_SEASONS = list(range(START_YEAR, now.year + 1))
# MLB uses calendar years
MLB_YEARS = list(range(START_YEAR, now.year + 1))

print(f"Date range: {DATE_START_LOCAL.date()} → {DATE_END_LOCAL.date()}")
print(f"ESPN seasons: {ESPN_SEASONS[0]}–{ESPN_SEASONS[-1]} | MLB years: {MLB_YEARS[0]}–{MLB_YEARS[-1]}")

# ---- Venue mapping ----
VENUE_MAP = {
    "Madison Square Garden": {
        "venue_borough": "Manhattan",
        "venue_taxi_zone_hint": "Midtown / Penn Station / Madison Square Garden",
        "venue_pickup_location_id": None,
        "venue_latitude": 40.7505,
        "venue_longitude": -73.9934,
        "include_in_taxi_analysis": True
    },
    "Barclays Center": {
        "venue_borough": "Brooklyn",
        "venue_taxi_zone_hint": "Prospect Heights / Atlantic Avenue",
        "venue_pickup_location_id": None,
        "venue_latitude": 40.6826,
        "venue_longitude": -73.9754,
        "include_in_taxi_analysis": True
    },
    "Yankee Stadium": {
        "venue_borough": "Bronx",
        "venue_taxi_zone_hint": "Yankee Stadium",
        "venue_pickup_location_id": None,
        "venue_latitude": 40.8296,
        "venue_longitude": -73.9262,
        "include_in_taxi_analysis": True
    },
    "Citi Field": {
        "venue_borough": "Queens",
        "venue_taxi_zone_hint": "Flushing / Willets Point",
        "venue_pickup_location_id": None,
        "venue_latitude": 40.7571,
        "venue_longitude": -73.8458,
        "include_in_taxi_analysis": True
    },
}

LEAGUE_DURATION_HOURS = {
    "NBA": 2.5,
    "NHL": 2.5,
    "MLB": 3.0,
}

# ---- Helper functions ----

def parse_espn_datetime(dt_str):
    """Parse ESPN's ISO datetime string to UTC datetime."""
    if dt_str is None:
        return None
    s = dt_str.replace("Z", "+00:00")
    return datetime.fromisoformat(s).astimezone(timezone.utc)

def to_local(dt_utc):
    """Convert UTC datetime to NY local time."""
    if dt_utc is None:
        return None
    return dt_utc.astimezone(NY_TZ)

def in_date_range(dt_local):
    """Check if datetime falls within our configured date range."""
    return dt_local is not None and DATE_START_LOCAL <= dt_local < DATE_END_LOCAL

def normalize_event_id(*parts):
    """Generate a deterministic event ID from component parts."""
    raw = "||".join([str(p) for p in parts if p is not None])
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

# ---- ESPN API (NBA, NHL) ----

def espn_team_schedule(sport, league, team_alias, season):
    url = f"https://site.api.espn.com/apis/site/v2/sports/{sport}/{league}/teams/{team_alias}/schedule"
    params = {"season": season}
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    return response.json()

def extract_espn_events(payload, league_name, team_name, expected_venue_name, season):
    events = []
    for item in payload.get("events", []):
        competitions = item.get("competitions", [])
        if not competitions:
            continue
        comp = competitions[0]
        event_date_utc = parse_espn_datetime(item.get("date") or comp.get("date"))
        event_date_local = to_local(event_date_utc)
        if not in_date_range(event_date_local):
            continue

        # Identify home/away competitors
        competitors = comp.get("competitors", [])
        home = None
        away = None
        for c in competitors:
            if c.get("homeAway") == "home":
                home = c
            elif c.get("homeAway") == "away":
                away = c
        if not home:
            continue

        home_team_name = home.get("team", {}).get("displayName")
        away_team_name = away.get("team", {}).get("displayName") if away else None

        # Only home games for our target team
        if home_team_name != team_name:
            continue

        # Resolve venue
        venue = comp.get("venue", {}) or {}
        venue_name = venue.get("fullName") or expected_venue_name
        if expected_venue_name and expected_venue_name.lower() in (venue_name or "").lower():
            venue_name = expected_venue_name
        venue_info = VENUE_MAP.get(venue_name, VENUE_MAP.get(expected_venue_name, {}))

        duration_hours = LEAGUE_DURATION_HOURS[league_name]
        estimated_end_local = event_date_local + timedelta(hours=duration_hours)
        event_id = item.get("id") or normalize_event_id(league_name, team_name, away_team_name, event_date_local)

        events.append({
            "event_id": f"{league_name.lower()}_{event_id}",
            "event_source": "espn_site_api",
            "league": league_name,
            "season": season,
            "team_name": team_name,
            "opponent_name": away_team_name,
            "is_home_event": True,
            "event_name": f"{team_name} vs {away_team_name}",
            "event_type": "Sports",
            "venue_name": venue_name,
            "venue_borough": venue_info.get("venue_borough"),
            "venue_taxi_zone_hint": venue_info.get("venue_taxi_zone_hint"),
            "venue_pickup_location_id": venue_info.get("venue_pickup_location_id"),
            "venue_latitude": venue_info.get("venue_latitude"),
            "venue_longitude": venue_info.get("venue_longitude"),
            "event_start_ts_utc": event_date_utc.isoformat(),
            "event_start_ts_local": event_date_local.isoformat(),
            "event_date_local": event_date_local.date().isoformat(),
            "event_hour_local": event_date_local.hour,
            "estimated_event_end_ts_local": estimated_end_local.isoformat(),
            "event_duration_hours": duration_hours,
            "analysis_before_hours": 4,
            "analysis_after_hours": 4,
            "include_in_taxi_analysis": venue_info.get("include_in_taxi_analysis", True),
            "source_url": "https://site.api.espn.com/"
        })
    return events

# ---- MLB Stats API (Yankees, Mets) ----

def mlb_schedule(team_id, start_date, end_date):
    url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {
        "sportId": 1,
        "teamId": team_id,
        "startDate": start_date,
        "endDate": end_date,
        "hydrate": "team,venue",
    }
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    return response.json()

def extract_mlb_events(payload, team_id, team_name, expected_venue_name, season_year):
    events = []
    for d in payload.get("dates", []):
        for game in d.get("games", []):
            game_dt_str = game.get("gameDate")
            if not game_dt_str:
                continue
            event_date_utc = datetime.fromisoformat(game_dt_str.replace("Z", "+00:00")).astimezone(timezone.utc)
            event_date_local = to_local(event_date_utc)
            if not in_date_range(event_date_local):
                continue

            teams = game.get("teams", {})
            home_team = teams.get("home", {}).get("team", {})
            away_team = teams.get("away", {}).get("team", {})
            home_team_id = home_team.get("id")
            away_team_name = away_team.get("name")

            # Only home games
            if int(home_team_id) != int(team_id):
                continue

            # Resolve venue
            venue = game.get("venue", {}) or {}
            venue_name = venue.get("name") or expected_venue_name
            if expected_venue_name and expected_venue_name.lower() in (venue_name or "").lower():
                venue_name = expected_venue_name
            venue_info = VENUE_MAP.get(venue_name, VENUE_MAP.get(expected_venue_name, {}))

            duration_hours = LEAGUE_DURATION_HOURS["MLB"]
            estimated_end_local = event_date_local + timedelta(hours=duration_hours)
            game_pk = game.get("gamePk") or normalize_event_id("MLB", team_name, away_team_name, event_date_local)

            events.append({
                "event_id": f"mlb_{game_pk}",
                "event_source": "mlb_stats_api",
                "league": "MLB",
                "season": season_year,
                "team_name": team_name,
                "opponent_name": away_team_name,
                "is_home_event": True,
                "event_name": f"{team_name} vs {away_team_name}",
                "event_type": "Sports",
                "venue_name": venue_name,
                "venue_borough": venue_info.get("venue_borough"),
                "venue_taxi_zone_hint": venue_info.get("venue_taxi_zone_hint"),
                "venue_pickup_location_id": venue_info.get("venue_pickup_location_id"),
                "venue_latitude": venue_info.get("venue_latitude"),
                "venue_longitude": venue_info.get("venue_longitude"),
                "event_start_ts_utc": event_date_utc.isoformat(),
                "event_start_ts_local": event_date_local.isoformat(),
                "event_date_local": event_date_local.date().isoformat(),
                "event_hour_local": event_date_local.hour,
                "estimated_event_end_ts_local": estimated_end_local.isoformat(),
                "event_duration_hours": duration_hours,
                "analysis_before_hours": 4,
                "analysis_after_hours": 4,
                "include_in_taxi_analysis": venue_info.get("include_in_taxi_analysis", True),
                "source_url": "https://statsapi.mlb.com/api/v1/schedule"
            })
    return events

# ---- Acquire events across all seasons ----

sports_events = []

# ESPN: NBA + NHL
ESPN_TEAMS = [
    {"league": "NBA", "sport": "basketball", "espn_league": "nba", "team_alias": "nyk", "team_name": "New York Knicks", "venue_name": "Madison Square Garden"},
    {"league": "NBA", "sport": "basketball", "espn_league": "nba", "team_alias": "bkn", "team_name": "Brooklyn Nets", "venue_name": "Barclays Center"},
    {"league": "NHL", "sport": "hockey", "espn_league": "nhl", "team_alias": "nyr", "team_name": "New York Rangers", "venue_name": "Madison Square Garden"},
]

for t in ESPN_TEAMS:
    team_total = 0
    for season in ESPN_SEASONS:
        try:
            payload = espn_team_schedule(
                sport=t["sport"],
                league=t["espn_league"],
                team_alias=t["team_alias"],
                season=season
            )
            extracted = extract_espn_events(
                payload=payload,
                league_name=t["league"],
                team_name=t["team_name"],
                expected_venue_name=t["venue_name"],
                season=season
            )
            sports_events.extend(extracted)
            team_total += len(extracted)
            time.sleep(0.3)
        except Exception as e:
            print(f"  Warning: {t['team_name']} season {season}: {e}")
    print(f"{t['team_name']}: {team_total} home games")

# MLB: Yankees + Mets
MLB_TEAMS = [
    {"team_id": 147, "team_name": "New York Yankees", "venue_name": "Yankee Stadium"},
    {"team_id": 121, "team_name": "New York Mets", "venue_name": "Citi Field"},
]

for t in MLB_TEAMS:
    team_total = 0
    for year in MLB_YEARS:
        try:
            payload = mlb_schedule(
                team_id=t["team_id"],
                start_date=f"{year}-01-01",
                end_date=f"{year}-12-31"
            )
            extracted = extract_mlb_events(
                payload=payload,
                team_id=t["team_id"],
                team_name=t["team_name"],
                expected_venue_name=t["venue_name"],
                season_year=year
            )
            sports_events.extend(extracted)
            team_total += len(extracted)
            time.sleep(0.3)
        except Exception as e:
            print(f"  Warning: {t['team_name']} {year}: {e}")
    print(f"{t['team_name']}: {team_total} home games")

# ---- Deduplicate, validate, and write ----

# Dedup by event_id (same game may appear in overlapping season queries)
deduped = {}
for e in sports_events:
    deduped[e["event_id"]] = e

sports_events = sorted(deduped.values(), key=lambda x: x["event_start_ts_local"])
print(f"\nTotal events after dedup: {len(sports_events)}")

if len(sports_events) == 0:
    raise ValueError("No sports events found. Aborting.")

# Write to temp file first (atomic replace pattern)
with open(sports_tmp_file, "w") as f:
    for row in sports_events:
        f.write(json.dumps(row) + "\n")

# Validate temp file is readable by Spark
tmp_df = spark.read.json(sports_adls_tmp_file)
tmp_count = tmp_df.count()
print(f"Validated temp file: {tmp_count} rows")

if tmp_count == 0:
    raise ValueError("Temp file is empty after Spark read. Aborting.")

# Promote: remove old final, move temp to final
try:
    dbutils.fs.rm(sports_adls_final_file)
except Exception:
    pass

dbutils.fs.mv(sports_adls_tmp_file, sports_adls_final_file)
print(f"Promoted to final: events/sports_events.json")

# Cleanup legacy dated files from earlier bootstrap versions
for old_name in os.listdir(sports_target_dir):
    if old_name.startswith("sports_events_") and old_name.endswith(".json"):
        old_path = os.path.join(sports_target_dir, old_name)
        try:
            os.remove(old_path)
            dbutils.fs.rm(f"{LANDING_PATH}/events/{old_name}")
            print(f"Removed legacy file: {old_name}")
        except Exception:
            pass

print(f"\nDone. {len(sports_events)} events: {sports_events[0]['event_date_local']} → {sports_events[-1]['event_date_local']}")

In [0]:
# Validate sports events in landing zone

sports_landing_path = f"{LANDING_PATH}/events/sports_events.json"
sports_test_df = spark.read.json(sports_landing_path)

print(f"Total sports events: {sports_test_df.count()}")

# Summary by league and team
display(
    sports_test_df
    .groupBy("league", "team_name", "venue_name")
    .count()
    .orderBy("league", "team_name")
)

# Distribution by year and league
from pyspark.sql.functions import year as yr, col
display(
    sports_test_df
    .withColumn("event_year", yr(col("event_date_local").cast("date")))
    .groupBy("event_year", "league")
    .count()
    .orderBy("event_year", "league")
)